# EDA — BUECHEREIOGD.csv (Büchereien / libraries)

City of Vienna open data export. Source file: `data/raw/BUECHEREIOGD.csv`. Run all cells to
reproduce the findings below.

In [1]:
import pandas as pd
import re

df = pd.read_csv("../data/raw/BUECHEREIOGD.csv")
df.shape

(37, 16)

## Columns & dtypes

In [2]:
df.dtypes

FID                     str
SHAPE                   str
NAME                    str
ADRESSE                 str
OEFFNUNGSZEITEN1        str
OEFFNUNGSZEITEN2        str
OEFFNUNGSZEITEN3        str
OEFFNUNGSZEITEN4        str
OEFFNUNGSZEITEN5        str
OEFFNUNGSZEITEN6        str
TELEFON                 str
EMAIL                   str
WEBLINK1                str
BEZIRK                int64
SE_SDO_ROWID          int64
SE_ANNO_CAD_DATA    float64
dtype: object

## Missing values

In [3]:
df.isnull().sum()

FID                  0
SHAPE                0
NAME                 0
ADRESSE              0
OEFFNUNGSZEITEN1     0
OEFFNUNGSZEITEN2     0
OEFFNUNGSZEITEN3     0
OEFFNUNGSZEITEN4     0
OEFFNUNGSZEITEN5    26
OEFFNUNGSZEITEN6    36
TELEFON              0
EMAIL                1
WEBLINK1             0
BEZIRK               0
SE_SDO_ROWID         0
SE_ANNO_CAD_DATA    37
dtype: int64

## Duplicate check

In [4]:
print("Duplicate full rows:", df.duplicated().sum())
print("Duplicate NAME:", df["NAME"].duplicated().sum())

Duplicate full rows: 0
Duplicate NAME: 0


## Parse geometry

`SHAPE` is a WKT geometry string. Parse into `lon`/`lat` (works for the first
coordinate pair even if the geometry is a line/polygon) and sanity-check the
range against Vienna's bounding box.

In [5]:
def parse_first_point(s):
    m = re.search(r"(-?\d+\.\d+)\s+(-?\d+\.\d+)", str(s))
    if m:
        return float(m.group(1)), float(m.group(2))
    return None, None

df["lon"], df["lat"] = zip(*df["SHAPE"].map(parse_first_point))
print("Unparseable SHAPE values:", df["lon"].isnull().sum())
print("lon range:", df["lon"].min(), "-", df["lon"].max())
print("lat range:", df["lat"].min(), "-", df["lat"].max())

Unparseable SHAPE values: 0
lon range: 16.27103027040828 - 16.509260024935404
lat range: 48.13567411991216 - 48.278310429288744


## District (`BEZIRK`) distribution

In [6]:
df["BEZIRK"].value_counts(dropna=False).sort_index()

BEZIRK
2     2
3     3
4     1
5     1
6     1
7     2
9     1
10    2
11    2
12    2
13    1
14    2
15    2
16    2
17    1
18    1
19    2
20    2
21    3
22    2
23    2
Name: count, dtype: int64

## Opening hours & contact fields

Six `OEFFNUNGSZEITEN1..6` free-text columns (one per opening-hours block), plus
`TELEFON`/`EMAIL`/`WEBLINK1`. Check how many blocks are actually used and what the
text format looks like.

In [7]:
oeff_cols = [c for c in df.columns if c.startswith("OEFFNUNGSZEITEN")]
print("non-null count per opening-hours column:")
print(df[oeff_cols].notnull().sum())
print("\nsample OEFFNUNGSZEITEN1 values:")
print(df["OEFFNUNGSZEITEN1"].head(5).tolist())
print("\nEMAIL missing:", df["EMAIL"].isnull().sum(), "of", len(df))

non-null count per opening-hours column:
OEFFNUNGSZEITEN1    37
OEFFNUNGSZEITEN2    37
OEFFNUNGSZEITEN3    37
OEFFNUNGSZEITEN4    37
OEFFNUNGSZEITEN5    11
OEFFNUNGSZEITEN6     1
dtype: int64

sample OEFFNUNGSZEITEN1 values:
['Mo 11.00 bis 19.00 Uhr', 'Mo 11.00 bis 18.00 Uhr', 'Mo 10.00 bis 12.00 Uhr und 13.00 bis 18.00 Uhr', 'Mo 10.00 bis 12.00 Uhr und 13.00 bis 18.00 Uhr', 'Mo 11.00 bis 19.00 Uhr']

EMAIL missing: 1 of 37


## Sample rows

In [8]:
df[["NAME", "BEZIRK", "ADRESSE", "lon", "lat"]].sample(5, random_state=1)

,NAME,BEZIRK,ADRESSE,lon,lat
2,Bücherei Margareten,5,"5., Pannaschgasse 6",16.354783,48.188616
29,Bücherei Hernals,17,"17., Hormayrgasse 2",16.331132,48.218737
3,Bücherei Mariahilf,6,"6., Gumpendorferstraße 59-61",16.354391,48.197607
22,Bücherei Hietzing,13,"13., Preyergasse 1-7",16.284914,48.174110
25,Bücherei Schwendermarkt,15,"15., Schwendergasse 39-43",16.324876,48.190313


## Findings

**Basics:** 37 records, 16 columns. Clean dataset, no duplicates (full-row or
`NAME`).

**Missing values:** `OEFFNUNGSZEITEN5`/`6` are null for most rows (26/37 and
36/37) — most libraries only need up to 4 opening-hours blocks, the extra columns
exist for the few with more complex schedules. `EMAIL` missing for 1 row.

**Opening hours are free text** (e.g. `"Mo 11.00 bis 19.00 Uhr"`), not structured
— fine to store as a display string, but not directly queryable for "is this open
right now" reasoning without parsing. Worth flagging as a stretch-goal parsing
task rather than in-scope for the POC.

**Geometry:** clean WGS84 points, has both `BEZIRK` and structured `ADRESSE`.

**Suitability for the KG:** good candidate, same shape as `MUSEUMOGD.csv`.
Suggested mapping: `NAME` → `poi:name`, `BEZIRK` → `poi:district`, `ADRESSE` →
`poi:address`, `lon`/`lat` → `geo:long`/`geo:lat`, opening-hours columns → a single
concatenated `poi:openingHours` text property (unless hours-based reasoning
becomes in-scope later), `TELEFON`/`EMAIL`/`WEBLINK1` → optional contact
properties.